# SAFE ML Txns Endpoint

This notebook builds a clustering model (K-Means) and publishes it as an Amazon SageMaker endpoint, as well as testing it with data in S3 and saving the predictions. It is designed for batch/near-real-time flows within the blossom-analytics-safe-dev-nv bucket.

In [ ]:
import boto3
import os

### (Don't run) Step 1: Prepare the file **model.tar.gz**

Download the files needed for inference from S3 and packaging for SageMaker by create model.tar.gz with all artifacts 

* preprocessing_pipeline.joblib (preprocessing pipeline)
* selected_features.csv (features used)
* kmeans_model.joblib (model)
* centroids.csv (clustering centroids)
* kmeans_artifacts.json (metadata)

In [ ]:
import tarfile

files_to_download = {
    "kmeans_model.joblib": "output/kmeans/kmeans_analysis/artifact/kmeans_model.joblib",  #artifacts/kmeans_model.joblib
    "preprocessing_pipeline.joblib": "output/preprocessing/preprocessing_pipeline.joblib",  
    "selected_features.csv": "output/feature_selection/selected_features.csv",
    "centroids.csv": "output/kmeans/centroids/centroids.csv",
    "kmeans_artifacts.json": "output/kmeans/kmeans_analysis/artifact/kmeans_artifacts.json"
}

bucket_name = 'blossom-analytics-safe-dev-nv' ## definir con Ramos
local_dir = "github/safe_txns_sim_endpoint/endpoint/model_files" ## a definir carpetas locales
os.makedirs(local_dir, exist_ok=True)

s3 = boto3.client("s3")

# Download each file
for local_name, s3_key in files_to_download.items():
    local_path = os.path.join(local_dir, local_name)
    print(f"Downloading s3://{bucket_name}/{s3_key} → {local_path}")
    s3.download_file(bucket_name, s3_key, local_path)

# Create model.tar.gz
tar_path = "github/safe_txns_sim_endpoint/endpoint/model.tar.gz"
with tarfile.open(tar_path, "w:gz") as tar:
    for filename in files_to_download.keys():
        tar.add(os.path.join(local_dir, filename), arcname=filename)

print(f"File generated: {tar_path}")

### (Don't run) Step 2: Upload model.tar.gz to S3

Upload model.tar.gz with all artifacts to S3 at s3://blossom-analytics-safe-dev-nv/output/

In [ ]:
model_key = "safe_txns/similarity/endpoint/v0/model.tar.gz"  # a definir nueva ruta en S3
s3.upload_file("github/safe_txns_sim_endpoint/endpoint/model.tar.gz", bucket_name, model_key)
model_artifact_uri = f"s3://{bucket_name}/{model_key}"
print("S3 URI for SKLearnModel:", model_artifact_uri)

### Step 3: Deploy the endpoint with SKLearnModel

V2: Previous Endpoint
* **model_artifact_uri** = "s3://blossom-analytics-safe-dev-nv/output/kmeans-endpoint-final/model.tar.gz"
* **entry_point** = "new-inference.py"
* **endpoint_name** = "final-safe-txns-endpoint"

V3: Final Endpoint 
* **model_artifact_uri** = "s3://blossom-analytics-safe-dev-nv/output/kmeans-endpoint-last3/model.tar.gz"
* **entry_point** = "new-inference-last.py"
* **endpoint_name** = "data-safe-txns-endpoint"

In [ ]:
from sagemaker.sklearn.model import SKLearnModel
from sagemaker import get_execution_role, Session
import sagemaker

# Setup
sagemaker_session = sagemaker.Session()
role = get_execution_role()

# S3 path
#!!!!
#V2
# model_artifact_uri = "s3://blossom-analytics-safe-dev-nv/output/kmeans-endpoint-final/model.tar.gz"  # Previous endpoint (V2)
#V3
model_artifact_uri = "s3://blossom-analytics-safe-dev-nv/safe_txns/similarity/endpoint/v0/model.tar.gz" # Final endpoint (V3)

# Create the model
sk_model = SKLearnModel(
    model_data=model_artifact_uri,
    role=role,
    entry_point="inference_rules.py",
    source_dir="github/safe_txns_sim_endpoint/endpoint",
    framework_version="1.2-1",
    sagemaker_session=sagemaker_session
)

In [ ]:
# Deploy the endpoint
predictor = sk_model.deploy(
    initial_instance_count=1,
    instance_type="ml.m5.large",
    endpoint_name="data-safe-txns-sim-endpoint"
)

### Step 4: Test the endpoint from your local machine

In [ ]:
import boto3
import pandas as pd
import io
import json

In [ ]:
# Parameters
#change with the name of your input file and desired output file
data = "test.csv"
input_key = f"safe_txns/similarity/real_time/input/{data}" #new-real-time-example-rename.csv 
output_key = f"safe_txns/similarity/real_time/output/{data}"

In [ ]:
%%time
#!!!!
endpoint_name = "data-safe-txns-sim-endpoint" 
#V2: final-safe-txns-endpoint
#V3: data-safe-txns-endpoint

region = "us-east-1"
bucket_name = 'blossom-analytics-safe-dev-nv'

# Download CSV from S3
s3 = boto3.client("s3")
response = s3.get_object(Bucket=bucket_name, Key=input_key)
df = pd.read_csv(response["Body"])
print(f"File loaded with {len(df)} rows")

# Convert to CSV in memory (header=True, index=False)
csv_buffer = io.StringIO()
df.to_csv(csv_buffer, header=True, index=False)
payload = csv_buffer.getvalue()

# Invoke the endpoint
runtime = boto3.client("sagemaker-runtime", region_name=region)
response = runtime.invoke_endpoint(
    EndpointName=endpoint_name,
    ContentType="text/csv",
    Body=payload
)

# Read and parse results
result = response['Body'].read().decode('utf-8')
parsed = json.loads(result)

In [ ]:
parsed[0] # Endpoint Similarity

In [ ]:
parsed[0] # Endpoint Original

In [ ]:
# Convert to DataFrame
df_result = pd.DataFrame(parsed)

# Save in S3
output_buffer = io.StringIO()
df_result.to_csv(output_buffer, index=False)
s3.put_object(Bucket=bucket_name, Key=output_key, Body=output_buffer.getvalue())

print(f"Result saved in s3://{bucket_name}/{output_key}")

### ! Delete endpoint

In [ ]:
---

In [ ]:
client = boto3.client("sagemaker")
endpoint_config_name = "data-safe-txns-sim-endpoint" #data-safe-txns-similarity-endpoint

try:
    client.delete_endpoint_config(EndpointConfigName=endpoint_config_name)
    print(f"EndpointConfig '{endpoint_config_name}' deleted.")
except client.exceptions.ClientError as e:
    if "Could not find endpoint configuration" in str(e):
        print("EndpointConfig does not exist, continuing.")
    else:
        raise

In [ ]:
# ! Force delete
client = boto3.client("sagemaker", region_name="us-east-1")
client.delete_endpoint(EndpointName=endpoint_config_name)
#client.delete_endpoint_config(EndpointConfigName=endpoint_config_name)

#### Additional: Check Endpoint status

In [ ]:
import time, boto3
sm = boto3.client("sagemaker")

EP = "data-safe-txns-endpoint"   # o el -v2 si lo cambiaste

for _ in range(120):
    d = sm.describe_endpoint(EndpointName=EP)
    st = d["EndpointStatus"]
    print("status:", st)
    if "FailureReason" in d and d["FailureReason"]:
        print("FailureReason:", d["FailureReason"])   # clave: aquí suele venir el stacktrace del contenedor
        break
    if st in ("InService", "Failed"):
        break
    time.sleep(10)